# Lexical Fields Analysis

The goal of this notebook is to analyze characteristics of the three lexical fields (`title`, `topics`, `keywords`), which will be useful to finalize lexical retrieval design.

In [2]:
import duckdb as db
import pandas as pd
import matplotlib.pyplot as plt

from scholar_rank.utils import PROJECT_ROOT
COMPACT_PATH = PROJECT_ROOT/'data'/'compact'

In [ ]:
'''
    Checking keywords count distribution.
'''

con = db.connect()

rel = con.read_parquet(f"{str(COMPACT_PATH)}/works/**/*.parquet")

result = con.sql("""
    SELECT len(keywords) as keywords_count, COUNT(*) as freq
    from rel
    GROUP BY keywords_count
    ORDER BY keywords_count
""").df()

,keywords_count,freq
0,0,39275901
1,1,48460467
2,2,25228494
3,3,61043241
4,4,67394289


In [ ]:
result["pct"] = (result["freq"]/result["freq"].sum()*100).round(3)
result

,keywords_count,freq,pct
0,0,39275901,7.695531e+00
1,1,48460467,9.495111e+00
2,2,25228494,4.943150e+00
3,3,61043241,1.196052e+01
4,4,67394289,1.320491e+01
5,5,64119215,1.256321e+01
6,6,31859861,6.242468e+00
7,7,22005335,4.311620e+00
8,8,23737543,4.651020e+00
9,9,24570189,4.814165e+00


In [ ]:
'''
    Topic score distribution by rank (primary/2nd/3rd topic).

    Single-pass aggregate query (stats computed in DuckDB, not fetched row-by-row into
    pandas) -- an earlier version of this used UNION ALL per rank, which re-scans the
    full 207GB corpus once per rank (3-5x the necessary I/O) and timed out at 10 minutes.
    This version scans once.
'''

def stats_expr(col, alias):
    return f"""
        count({col}) AS n_{alias}, avg({col}) AS mean_{alias}, stddev({col}) AS std_{alias},
        min({col}) AS min_{alias}, quantile_cont({col}, 0.25) AS q25_{alias},
        quantile_cont({col}, 0.5) AS median_{alias}, quantile_cont({col}, 0.75) AS q75_{alias},
        max({col}) AS max_{alias}
    """

topic_rank_stats = con.sql(
    "SELECT " + ", ".join(stats_expr(f"topics[{i}].score", f"rank{i}") for i in [1, 2, 3]) + " FROM rel"
).df()
topic_rank_stats

In [ ]:
'''
    Keyword score distribution by rank, split by whether the document has an abstract.

    OpenAlex computes keyword similarity against title+abstract (BGE M3 embeddings) --
    checking whether that produces systematically different scores when abstract is
    null (59.1% of the corpus) vs present, which matters for whether raw keyword score
    is safe to use as a cross-document weight.
'''

def stats_expr_filtered(col, alias, cond):
    return f"""
        count({col}) FILTER (WHERE {cond}) AS n_{alias},
        avg({col}) FILTER (WHERE {cond}) AS mean_{alias},
        stddev({col}) FILTER (WHERE {cond}) AS std_{alias},
        quantile_cont({col}, 0.5) FILTER (WHERE {cond}) AS median_{alias}
    """

parts = []
for i in [1, 2, 3, 4, 5]:
    parts.append(stats_expr_filtered(f"keywords[{i}].score", f"rank{i}_noabs", "abstract_inverted_index IS NULL"))
    parts.append(stats_expr_filtered(f"keywords[{i}].score", f"rank{i}_abs", "abstract_inverted_index IS NOT NULL"))

keyword_rank_stats = con.sql("SELECT " + ", ".join(parts) + " FROM rel").df()
keyword_rank_stats

In [ ]:
'''
    Topic hierarchy convergence: for documents with 2+ topics, how often do those
    topics share the same subfield/field/domain? Relevant to whether NOT deduplicating
    shared hierarchy labels (see docs/retrieval_engine.md discussion) preserves a real
    concentration signal, and at which hierarchy level that signal is actually useful.

    Note: subfield_id/field_id/domain_id are flat fields on the topic struct (not
    nested, e.g. NOT topics[i].subfield.id) -- extract_compact already flattened them.
'''

convergence = con.sql("""
    SELECT
        avg(CASE WHEN n_distinct_subfields < n_topics THEN 1.0 ELSE 0.0 END) * 100 AS subfield_convergence_pct,
        avg(CASE WHEN n_distinct_fields < n_topics THEN 1.0 ELSE 0.0 END) * 100 AS field_convergence_pct,
        avg(CASE WHEN n_distinct_domains < n_topics THEN 1.0 ELSE 0.0 END) * 100 AS domain_convergence_pct,
        count(*) AS n_multi_topic_docs
    FROM (
        SELECT
            len(topics) AS n_topics,
            len(list_distinct(list_transform(topics, t -> t.subfield_id))) AS n_distinct_subfields,
            len(list_distinct(list_transform(topics, t -> t.field_id))) AS n_distinct_fields,
            len(list_distinct(list_transform(topics, t -> t.domain_id))) AS n_distinct_domains
        FROM rel
        WHERE len(topics) >= 2
    )
""").df()
convergence

## Findings (full corpus, run 2026-07-23)

**Topic score by rank** — magnitude drops sharply and monotonically with rank, not just slightly:

| rank | n | mean | median | std |
|---|---|---|---|---|
| 1 (primary) | 395.1M | 0.618 | **0.801** | 0.391 |
| 2 | 344.5M | 0.472 | **0.195** | 0.442 |
| 3 | 311.4M | 0.429 | **0.074** | 0.453 |

Median more than quadruples from rank 3 to rank 1. This is a real, large effect, not noise — rank position
carries substantial magnitude information, more than a generic `1/(k+rank)` decay might assume. Worth
factoring in if/when raw-magnitude weighting gets revisited later (see rank-based vs. magnitude-based
discussion in `docs/retrieval_engine.md`).

**Keyword score by rank x abstract presence — confirms the concern directly, not just plausibly.** At every
rank (1-5), has-abstract documents score consistently, meaningfully higher than no-abstract documents:

| rank | no abstract (mean) | has abstract (mean) | gap |
|---|---|---|---|
| 1 | 0.510 | 0.636 | +0.126 |
| 2 | 0.426 | 0.534 | +0.108 |
| 3 | 0.373 | 0.466 | +0.093 |
| 4 | 0.375 | 0.435 | +0.060 |
| 5 | 0.377 | 0.431 | +0.054 |

This is a systematic, consistent bias, not sampling noise — keyword score is **not safe to compare directly
across the abstract-present/abstract-null split** without correction. Directly supports the rank-based
weighting recommendation over raw-magnitude weighting for keywords specifically: rank-based weighting only
depends on within-document ordering, which this confound doesn't touch (a document's own top keyword is
still its top keyword regardless of which group it's in), whereas raw-magnitude weighting would need explicit
per-group calibration to avoid systematically favoring has-abstract documents.

**Topic hierarchy convergence** (documents with 2+ topics, n=344.5M) — confirms convergence is common, and
reveals the levels differ a lot in how *informative* convergence is:

| level | convergence rate |
|---|---|
| subfield | 32.6% |
| field | 76.1% |
| domain | 94.3% |

Subfield-level convergence (~1/3 of multi-topic docs) is the most informative level for a "concentration"
signal — common enough to matter, rare enough to discriminate. Domain-level convergence is nearly universal
(94.3%) — with only ~4-5 domains total in OpenAlex's taxonomy, most multi-topic papers converging there is
close to a given, so it likely adds little ranking signal despite being real. Worth weighting subfield-level
convergence more than domain-level convergence if this becomes an explicit feature, rather than treating all
three levels as equally informative.

In [ ]:
'''
    Word overlap between a topic's own display_name and its subfield/field/domain
    display_name -- i.e. does indexing a topic's full hierarchy chain naturally repeat
    words, even for a single topic with no convergence with other topics involved?

    The topic taxonomy itself is small and fixed (~4,516 topics) so this only needs
    the DISTINCT combinations, not a full corpus scan of raw rows -- cheap relative to
    the earlier aggregate queries.
'''

import re

distinct_topics = con.sql("""
    WITH t AS (SELECT unnest(topics) AS x FROM rel)
    SELECT DISTINCT x.display_name, x.subfield_display_name, x.field_display_name, x.domain_display_name
    FROM t
""").fetchall()

STOPWORDS = {"and","the","of","for","in","on","to","a","an","its","with","from","by","or","as","at"}

def words(s):
    return set(w for w in re.sub(r'[^a-zA-Z0-9 ]', ' ', s.lower()).split() if w not in STOPWORDS and len(w) > 2)

pairs = ["topic-subfield", "topic-field", "topic-domain", "subfield-field", "subfield-domain", "field-domain"]
overlap_counts = {p: 0 for p in pairs}

for topic, subfield, field, domain in distinct_topics:
    wt, wsf, wf, wd = words(topic), words(subfield), words(field), words(domain)
    checks = {
        "topic-subfield": wt & wsf, "topic-field": wt & wf, "topic-domain": wt & wd,
        "subfield-field": wsf & wf, "subfield-domain": wsf & wd, "field-domain": wf & wd,
    }
    for p, overlap in checks.items():
        if overlap:
            overlap_counts[p] += 1

n = len(distinct_topics)
overlap_pct = pd.Series({p: round(100 * overlap_counts[p] / n, 1) for p in pairs}, name="pct_sharing_a_word")
overlap_pct

### Word overlap findings (n=4,516 distinct topic combinations — the full OpenAlex topic taxonomy)

| pair | % sharing a real content word | example |
|---|---|---|
| topic ↔ subfield | 23.5% | "Reconstructive Surgery and Microvascular Techniques" / "Surgery" → `surgery` |
| subfield ↔ field | 42.8% | "Pulmonary and Respiratory Medicine" / "Medicine" → `medicine` |
| field ↔ domain | 27.1% | "Social Sciences" / "Social Sciences" → **identical strings**, not just a shared word |
| topic ↔ field | 10.7% | "Historical Astronomy and Related Studies" / "Physics and Astronomy" → `astronomy` |
| topic ↔ domain | 4.6% | "Social and Political Issues" / "Social Sciences" → `social` |
| subfield ↔ domain | 5.6% | "Public Health, Environmental and Occupational Health" / "Health Sciences" → `health` |

**Pattern: overlap tracks hierarchy adjacency, not distance from the topic itself.** Adjacent-level pairs
(topic-subfield, subfield-field, field-domain) all land in the 23-43% range; non-adjacent pairs (topic-field,
topic-domain, subfield-domain) drop to 4.6-10.7%. OpenAlex's naming convention reuses terminology between
neighboring taxonomy levels much more than across distant ones.

One sub-case worth calling out separately: some field/domain pairs aren't just word-overlapping, they're
**literally identical strings** (e.g. "Social Sciences" is both the field name and the domain name for that
branch) — a stronger case than general word-sharing.

Caught and fixed one thing while building this: the first pass flagged "and" as an overlapping word between
categories — a stopword false positive (topic-field misread as 30.4% before the fix), not real content
overlap. Corrected by filtering a standard stopword list before comparing.

**Relevant to the BM25F field-weighting design** (`docs/retrieval_engine.md`, concern 2): this confirms
within-topic hierarchy word repetition is real even for a single topic with no cross-topic convergence
involved (the earlier LCA/convergence finding above was a separate, additional source of repetition). Same
reasoning applies as before — BM25's own saturation caps the benefit of repeated terms, so this doesn't
obviously need active stripping. Open question this data should inform, not settle on its own: whether
topic/subfield/field/domain get indexed as one combined field or as separate BM25F sub-fields.

In [ ]:
'''
    Abstract null-rate and its effect on IDF: since abstract is null for a large chunk
    of the corpus, any word confined to abstract text looks artificially rarer (and
    gets a higher BM25 IDF weight) than its true in-context commonness, because
    abstract-less documents trivially "don't contain" any abstract-only term. Checking
    this directly with "and" as a concrete example, rather than assuming IDF crushes
    common words to ~0 regardless of corpus composition.

    Also handles ~100 malformed/truncated abstract_inverted_index JSON values found
    while building this query -- TRY() skips them instead of raising.
'''

abstract_and_stats = con.sql("""
    SELECT
        count(*) AS n_total,
        count(*) FILTER (WHERE abstract_inverted_index IS NOT NULL) AS n_abstract,
        count(*) FILTER (WHERE abstract_inverted_index IS NOT NULL AND TRY(json_keys(abstract_inverted_index)) IS NULL) AS n_malformed,
        count(*) FILTER (WHERE list_contains(list_transform(TRY(json_keys(abstract_inverted_index)), x -> lower(x)), 'and')) AS n_and
    FROM rel
""").df()
abstract_and_stats

In [ ]:
import math

n_total = int(abstract_and_stats["n_total"][0])
n_abstract = int(abstract_and_stats["n_abstract"][0])
n_and = int(abstract_and_stats["n_and"][0])

idf_and = math.log((n_total - n_and + 0.5) / (n_and + 0.5) + 1)
doc_freq_full = n_and / n_total
doc_freq_abstract = n_and / n_abstract
abstract_null_rate = 1 - n_abstract / n_total

idf_and, doc_freq_full, doc_freq_abstract, abstract_null_rate

### Findings: abstract null-rate & IDF inflation (full corpus, run 2026-07-24)

| metric | value |
|---|---|
| Total documents | 510,372,821 |
| Non-null abstract | 266,931,548 (52.28%) |
| Abstract null rate | **47.72%** — corrects the earlier-logged 59.1%, which was measured on local shards before the full corpus pull completed |
| Malformed `abstract_inverted_index` JSON | 100 records (negligible, but real — `TRY()` needed, plain `json_keys()` raises) |
| Documents with "and" in abstract | 170,933,463 |
| "and" doc frequency, of full corpus | 33.5% |
| "and" doc frequency, of abstract-having docs only | 64.0% |
| **IDF("and")**, standard BM25 formula, `N`=full corpus | **≈1.094** |

**The IDF number is the interesting part.** Naive intuition says an ultra-common word like "and" should get
IDF ≈ 0 since it's in most documents that have text at all. That intuition is wrong for this corpus
specifically: IDF is computed against `N` = the *whole* 510M-document corpus, and 47.72% of that corpus has
no abstract at all — those documents trivially "don't contain" any abstract-only word, which deflates "and"'s
apparent document frequency (33.5% of everything, vs. 64.0% of documents that actually have abstract text)
and inflates its IDF accordingly. For comparison, a genuinely rare/topical term at 1% document frequency
scores IDF ≈ 4.6 — so "and" is weighted meaningfully below a real topical hit, but at ≈1.09 it's far from the
~0 a flat "IDF crushes stopwords" assumption would predict. Combined with BM25's saturation curve giving most
of its reward on a term's *first* occurrence, a single "and" in an abstract contributes a non-trivial amount
to the score (`IDF × saturated_tf ≈ 1.0-1.1` for `f=1`) — not the negligible amount it should logically be.

**This generalizes beyond "and."** The mechanism isn't "we forgot a stopword list" — it's that *any* word
common within abstract text but largely absent from the 47.72% null-abstract population will show the same
artificially elevated IDF (e.g. "results," "significant," "method," "using"). A stopword list catches the
canonical function words; it doesn't catch this broader class, because the distortion comes from the null
rate itself, not from any particular word being semantically empty.

**Decision this drove**: abstract is excluded from lexical (BM25) scoring for this phase — title + topics
(+ hierarchy) + keywords only. Logged in `CLAUDE.md` and referenced from `docs/retrieval_engine.md` §2
(concern 4). Separately, an earlier idea to "fix" the null-abstract case by redistributing field weight onto
the other fields per-document was explored and ruled out as unsound given BM25F's actual pooled-term-frequency
mechanics (no real weight budget to redistribute) — this abstract-exclusion decision is unrelated to that
dead end and stands on its own empirical grounds above.